# 08 - Build Product Search Qrels

- Build graded relevance pools for 30 fixed discovery queries.
- Keep LLM teacher labels as proxy judgments and preserve manual-review artifacts.

In [1]:
import os
from pathlib import Path
import sys
import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

from src.rag.config import load_project_config
from src.rag.evaluation.product_search_queries import assign_stratified_split
from src.rag.evaluation.product_search_runtime import load_product_search_evaluation_context
from src.rag.evaluation.product_search_qrels import ProductSearchQrelsBuilder

In [2]:
config = load_project_config(
    PROJECT_ROOT
)

cfg = config[
    "product_search"
][
    "evaluation"
]

queries = pd.DataFrame(
    cfg["queries"]
)

queries = assign_stratified_split(
    queries,
    test_fraction=cfg["test_fraction"],
    seed=cfg["seed"],
)

EVAL_DIR = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "product_search"
)

EVAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

queries.to_csv(
    EVAL_DIR / "queries.csv",
    index=False,
)

display(
    queries.groupby(
        ["query_type", "split"]
    ).size().unstack(
        fill_value=0
    )
)

print("Queries:", len(queries))

split,dev,test
query_type,,
attribute,4,2
brand,4,2
experiential,4,2
multi_constraint,4,2
negative,4,2


Queries: 30


In [3]:
context = load_product_search_evaluation_context(
    project_root=PROJECT_ROOT,
    api_key=os.environ["METIS_API_KEY"],
    base_url=os.environ["METIS_BASE_URL"],
)
pool_cfg = cfg["qrel_pool"]
builder = ProductSearchQrelsBuilder(
    metadata_retriever=context.metadata,
    review_evidence=context.review_evidence,
    judge=context.judge,
    hybrid_pool_k=pool_cfg["hybrid_k"],
    dense_pool_k=pool_cfg["dense_k"],
    sparse_pool_k=pool_cfg["sparse_k"],
    rescue_pool_k=pool_cfg["rescue_k"],
    rescue_fragment_k=pool_cfg["rescue_fragment_k"],
    rescue_max_fragments=pool_cfg["rescue_max_fragments"],
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [4]:
result = builder.build(
    queries=queries,
    qrels_path=EVAL_DIR / "qrels.parquet",
    telemetry_path=EVAL_DIR / "qrels_telemetry.csv",
    manual_review_path=EVAL_DIR / "qrels_manual_review.csv",
    resume=True,
)
qrels = result["qrels"]
telemetry = result["telemetry"]
print("Qrel rows:", len(qrels))
print("Queries judged:", qrels["query_id"].nunique())
display(qrels["teacher_grade"].value_counts().sort_index())
display(qrels["teacher_confidence"].value_counts())

Qrel rows: 454
Queries judged: 30


teacher_grade
0    247
1     66
2     81
3     60
Name: count, dtype: int64

teacher_confidence
high      397
medium     57
Name: count, dtype: int64

In [5]:
display(
    qrels[qrels["manual_review_priority"]][
        ["query_id", "query", "id", "title_fa", "teacher_grade", "teacher_confidence", "teacher_reason"]
    ].head(60)
)
print("Manual review CSV:", EVAL_DIR / "qrels_manual_review.csv")
print("Review at least the TEST rows before claiming independent human labels.")

,query_id,query,id,title_fa,teacher_grade,teacher_confidence,teacher_reason
0,q001,شامپو ضد ریزش سریتا,711943,شامپو ضد ریزش سریتا مدل پرومین مناسب برای موها...,3,high,شامپو ضد ریزش برند سریتا است؛ همراه داشتن تونی...
1,q001,شامپو ضد ریزش سریتا,6552521,شامپو تقویت کننده و ضد ریزش سریتا مدل Fortifyi...,3,high,شامپو تقویت‌کننده و ضد ریزش از برند سریتا است.
6,q001,شامپو ضد ریزش سریتا,8021274,شامپو ضد ریزش مو سریتا مدل مینوتا حجم 200 میلی...,3,high,مجموعه دو عددی شامپو ضد ریزش سریتا است.
7,q001,شامپو ضد ریزش سریتا,3832629,شامپو ضد ریزش مو سریتا مدل مینوتا مناسب برای م...,3,high,شامپو ضد ریزش سریتا است؛ بازخورد نیز رضایت و ک...
10,q001,شامپو ضد ریزش سریتا,471187,شامپو تقویت کننده و ضد ریزش سریتا مدل کافئین م...,3,high,شامپو تقویت‌کننده و ضد ریزش سریتا است؛ بازخورد...
16,q002,ضد آفتاب ژیناژن مناسب پوست چرب,12565854,کرم ضد آفتاب رنگی ژیناژن SPF 50 مدل 03 ‌مناسب ...,3,high,ضد آفتاب ژیناژن و صراحتاً مناسب پوست چرب است.
18,q002,ضد آفتاب ژیناژن مناسب پوست چرب,6733478,فلوئید ضد آفتاب بی رنگ الارو SPF50 مدل Ultra L...,1,high,ضد آفتاب مناسب پوست چرب است اما برند الارو است...
19,q002,ضد آفتاب ژیناژن مناسب پوست چرب,10985328,فلوئید ضد آفتاب بی رنگ نوکس SPF50 مدل LIGHT من...,1,high,برند نوکس است نه ژیناژن و نظر نیز چرب شدن و جو...
20,q002,ضد آفتاب ژیناژن مناسب پوست چرب,5428192,کرم ضد آفتاب ژیناژن مدل SPF50-02 مناسب پوست ها...,1,high,عنوان ژیناژن و پوست چرب را دارد، اما برند ثبت‌...
21,q002,ضد آفتاب ژیناژن مناسب پوست چرب,12565742,کرم ضد آفتاب رنگی ژیناژن SPF 50 مدل 02 ‌مناسب ...,3,high,کرم ضد آفتاب رنگی ژیناژن برای پوست چرب؛ نظر نی...


Manual review CSV: /home/ali/Desktop/projects/digikala-ai-assistant/data/evaluation/product_search/qrels_manual_review.csv
Review at least the TEST rows before claiming independent human labels.


## Evaluation-only recall rescue

- Add lexical rescue candidates only to the judged pool.
- Do not change production retrieval.

In [6]:
rescue_result = builder.augment_with_rescue(
    queries=queries,
    qrels_path=(
        EVAL_DIR
        / "qrels.parquet"
    ),
    telemetry_path=(
        EVAL_DIR
        / "qrels_telemetry.csv"
    ),
    manual_review_path=(
        EVAL_DIR
        / "qrels_manual_review.csv"
    ),
)

qrels = rescue_result["qrels"]
telemetry = rescue_result["telemetry"]

print(
    "New rescue qrel rows:",
    rescue_result[
        "new_qrel_rows"
    ],
)

print(
    "Total qrel rows:",
    len(qrels),
)

New rescue qrel rows: 133
Total qrel rows: 587


In [7]:
query_audit = (
    qrels.groupby(
        [
            "query_id",
            "query_type",
            "query",
            "split",
        ]
    )
    .agg(
        pool_size=(
            "id",
            "size",
        ),
        relevant_ge2=(
            "teacher_grade",
            lambda values: int(
                (
                    values
                    >= 2
                ).sum()
            ),
        ),
        rescue_candidates=(
            "rescue_pool_rank",
            lambda values: int(
                values.notna().sum()
            ),
        ),
    )
    .reset_index()
)

zero_positive = query_audit[
    query_audit[
        "relevant_ge2"
    ]
    == 0
]

print(
    "Queries with zero relevant products "
    "in judged pool:",
    len(zero_positive),
)

display(zero_positive)

display(
    qrels[
        qrels[
            "rescue_pool_rank"
        ].notna()
    ][
        [
            "query_id",
            "query",
            "id",
            "title_fa",
            "rescue_fragment",
            "rescue_pool_rank",
            "teacher_grade",
            "teacher_reason",
        ]
    ].head(80)
)

Queries with zero relevant products in judged pool: 8


,query_id,query_type,query,split,pool_size,relevant_ge2,rescue_candidates
3,q004,brand,پاوربانک شیائومی 20000,dev,20,0,0
4,q005,brand,هندزفری بی سیم انکر,test,21,0,5
8,q009,attribute,سرخ کن بدون روغن با ظرفیت حداقل 6 لیتر,test,19,0,6
9,q010,attribute,ماوس بی سیم ارگونومیک,test,21,0,5
14,q015,experiential,جارو شارژی سبک با مکش خوب,dev,19,0,5
22,q023,negative,پاوربانک که داغ نکنه,dev,16,0,0
26,q027,multi_constraint,هندزفری بلوتوث با باتری خوب و میکروفون مناسب تماس,dev,19,0,7
27,q028,multi_constraint,پاوربانک 20000 فست شارژ و سبک,dev,17,0,3


,query_id,query,id,title_fa,rescue_fragment,rescue_pool_rank,teacher_grade,teacher_reason
454,q001,شامپو ضد ریزش سریتا,894006,شامپو ضد ریزش موی کلینیک مدل 233,شامپو ضد ریزش,1.0,0,شامپو ضد ریزش است اما برند آن کلینیک است، نه س...
455,q001,شامپو ضد ریزش سریتا,370711,شامپو ضد ریزش مو درمالاین حجم 250 میلی لیتر,شامپو ضد ریزش,3.0,0,نوع محصول شامپو ضد ریزش است اما برند درمالاین ...
456,q001,شامپو ضد ریزش سریتا,3832551,تونیک ضد ریزش مو سریتا مدل مینوتا حجم 50 میلی ...,ضد ریزش سریتا,6.0,1,برند سریتا و کاربرد ضد ریزش دارد، اما تونیک مو...
457,q001,شامپو ضد ریزش سریتا,5408041,شامپو ضد شوره مو تگودر مدل ضد التهاب حجم 355 م...,شامپو ضد,7.0,0,شامپوی ضد شوره از برند تگودر است؛ نه ضد ریزش و...
458,q001,شامپو ضد ریزش سریتا,4344687,شامپو مو ایروکس مدل ضد شوره حجم 200 میلی لیتر ...,شامپو ضد,8.0,0,محصول شامل شامپوی ضد شوره و شامپو بدن ضد قارچ ...
...,...,...,...,...,...,...,...,...
529,q016,اسپیکر با صدای شفاف و بیس خوب,12385814,قطره چکان مدل شفاف,شفاف بیس,7.0,0,قطره‌چکان پزشکی است و ارتباطی با اسپیکر ندارد.
530,q016,اسپیکر با صدای شفاف و بیس خوب,10649965,سوت صدای مرغابی مدل حنا,صدای,8.0,0,سوت صدای مرغابی است، نه اسپیکر.
531,q017,کفش راحت برای پیاده روی طولانی,8854897,کفش پیاده روی دخترانه مدل Y-YEDS71,پیاده روی,5.0,2,نوع محصول کفش پیاده‌روی است و با کاربرد اصلی ج...
532,q017,کفش راحت برای پیاده روی طولانی,10794129,کفش مردانه کفش آداک مدل مجلسی,کفش,6.0,1,کفش رسمی مردانه است؛ با وجود اینکه کفش است، نو...
